In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

import warnings
warnings.filterwarnings("ignore")

# 1. READ & PREPROCESS
df = pd.read_csv(r"C:\Users\Ayush\Desktop\surge\kuppam\Hourly_Data_220kV.csv") 

df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], errors='coerce')
df = df.dropna(subset=['Datetime'])
df = df.drop_duplicates(subset=['Datetime'], keep='first')
df.set_index('Datetime', inplace=True)
df['Total Load (MW)'] = df['Total Load (MW)'].astype(str).str.replace(',', '')
df['Total Load (MW)'] = pd.to_numeric(df['Total Load (MW)'], errors='coerce')
df = df.asfreq('1H').ffill()

# 2. FEATURE SCALING
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[['Total Load (MW)']])

# 3. CREATE 3D SEQUENCES (168h past -> 24h future)
look_back = 168 
forecast_horizon = 24 

X, y = [], []
for i in range(look_back, len(df_scaled) - forecast_horizon + 1):
    X.append(df_scaled[i - look_back : i])
    y.append(df_scaled[i : i + forecast_horizon])

X = np.array(X)
y = np.array(y)

print(f"Shape of X (Encoder Input): {X.shape}")
print(f"Shape of y (Decoder Target): {y.shape}")

# 4. SPLIT (80/20)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# 5. BUILD SEQ2SEQ LSTM ARCHITECTURE
model = Sequential()
# ENCODER
model.add(LSTM(64, activation='relu', input_shape=(look_back, 1)))
# CONTEXT VECTOR BRIDGE
model.add(RepeatVector(forecast_horizon))
# DECODER
model.add(LSTM(64, activation='relu', return_sequences=True))
model.add(TimeDistributed(Dense(1)))

model.compile(optimizer='adam', loss='mse')

# 6. TRAIN MODEL
print("\nTraining Seq2Seq LSTM...")
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

# 7. PREDICT & UNSCALE
print("\nPredicting on Test Set...")
y_pred_scaled = model.predict(X_test)

y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(-1, forecast_horizon)
y_test_unscaled = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(-1, forecast_horizon)

# 8. METRICS & PLOT (Evaluating the 24-hour prediction chunks)
mae = mean_absolute_error(y_test_unscaled[:, 0], y_pred[:, 0])
rmse = np.sqrt(mean_squared_error(y_test_unscaled[:, 0], y_pred[:, 0]))
r2 = r2_score(y_test_unscaled[:, 0], y_pred[:, 0])

print("\n--- SEQ2SEQ LSTM PERFORMANCE ---")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

# Plotting a single random 24-hour future block from the test set
random_idx = 50 
plt.figure(figsize=(12, 5))
plt.plot(y_test_unscaled[random_idx], label='Actual 24h Load', color='black', marker='o')
plt.plot(y_pred[random_idx], label='LSTM Predicted 24h Load', color='red', linestyle='--', marker='x')
plt.title('Seq2Seq LSTM: Single Day-Ahead Forecast Wave')
plt.xlabel('Hours into the Future')
plt.ylabel('Load (MW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

C:\Users\Ayush\AppData\Local\Temp\ipykernel_13544\1552619239.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq('1H').ffill()


MAE : 9.4401
RMSE : 15.2777
R²: 0.8341
